# 08 - Stage 1 External PAN-12 Validation

This notebook applies the coded-chat Stage 1 model to PAN-12 test messages and compares predicted stage evidence against the existing line-level predatory ground truth. It does not use PAN-12 labels to retrain Stage 1.

Codes 200/600/900 are stage categories learned only from the coded chats. PAN-12 line labels indicate whether a message is predatory, so the check is indirect and must not be treated as stage ground truth.

In [1]:
from pathlib import Path
import re
import xml.etree.ElementTree as ET

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WORKSPACE = Path.cwd()
MODEL_PATH = WORKSPACE / "models" / "stage1_tfidf_logistic.joblib"
TEST_XML = WORKSPACE / "pan12-sexual-predator-identification-test-corpus-2012-05-17.xml"
TEST_LINE_LABELS = WORKSPACE / "pan12-sexual-predator-identification-groundtruth-problem2.txt"
REPORT_DIR = WORKSPACE / "reports"
PROCESSED_DIR = WORKSPACE / "data" / "processed"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for required in [MODEL_PATH, TEST_XML, TEST_LINE_LABELS]:
    if not required.exists():
        raise FileNotFoundError(f"Required input not found: {required}")

stage1_model = joblib.load(MODEL_PATH)

## Parse PAN-12 messages and line labels

Ground-truth problem 2 commonly stores records as `conversation_id line_number`. The parser treats both whitespace-separated and colon-separated records defensively.

In [2]:
DOCTYPE_PATTERN = re.compile(r"<!DOCTYPE[^>]*>", flags=re.IGNORECASE | re.DOTALL)
BARE_AMPERSAND_PATTERN = re.compile(r"&(?!#\d+;|#x[0-9A-Fa-f]+;|[A-Za-z_][A-Za-z0-9_.-]*;)")


def parse_xml(path: Path) -> ET.Element:
    raw = DOCTYPE_PATTERN.sub("", path.read_text(encoding="utf-8-sig"))
    raw = BARE_AMPERSAND_PATTERN.sub("&amp;", raw)
    return ET.fromstring(raw)


def load_pan12_messages(path: Path) -> pd.DataFrame:
    root = parse_xml(path)
    rows = []
    for conversation in root.findall(".//conversation"):
        conversation_id = conversation.attrib["id"]
        for message in conversation.findall("./message"):
            text_node = message.find("./text")
            author_node = message.find("./author")
            time_node = message.find("./time")
            rows.append({
                "conversation_id": conversation_id,
                "line": int(message.attrib["line"]),
                "author": "" if author_node is None or author_node.text is None else author_node.text.strip(),
                "time": "" if time_node is None or time_node.text is None else time_node.text.strip(),
                "text": "" if text_node is None or text_node.text is None else text_node.text.strip(),
            })
    return pd.DataFrame(rows)


def load_predatory_lines(path: Path) -> set[tuple[str, int]]:
    labels = set()
    for raw_line in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        parts = [part for part in re.split(r"[\s,:]+", line) if part]
        if len(parts) >= 2 and parts[0].strip() and parts[1].isdigit():
            labels.add((parts[0].strip(), int(parts[1])))
    return labels


messages = load_pan12_messages(TEST_XML)
predatory_lines = load_predatory_lines(TEST_LINE_LABELS)
messages["is_predatory_line"] = messages.apply(
    lambda row: int((row["conversation_id"], row["line"]) in predatory_lines), axis=1
)

print("PAN-12 test messages:", len(messages))
print("Ground-truth predatory message labels:", messages["is_predatory_line"].sum())

PAN-12 test messages: 2058781
Ground-truth predatory message labels: 6478


## Add context and Stage 1 probabilities

The same previous/target/next representation used by the baseline model is reconstructed. No PAN-12 ground truth participates in fitting.

In [3]:
messages = messages.sort_values(["conversation_id", "line"]).copy()
messages["previous_text"] = messages.groupby("conversation_id", sort=False)["text"].shift(1).fillna("")
messages["next_text"] = messages.groupby("conversation_id", sort=False)["text"].shift(-1).fillna("")
messages["model_text"] = (
    "previous: " + messages["previous_text"] + " target: " + messages["text"] + " next: " + messages["next_text"]
)

INFERENCE_BATCH_SIZE = 500
probability_batches = []
for batch_start in range(0, len(messages), INFERENCE_BATCH_SIZE):
    batch_text = messages["model_text"].iloc[batch_start:batch_start + INFERENCE_BATCH_SIZE]
    probability_batches.append(stage1_model.predict_proba(batch_text))
probabilities = np.vstack(probability_batches)
model_classes = [str(category) for category in stage1_model.classes_]
messages["predicted_stage"] = [model_classes[index] for index in probabilities.argmax(axis=1)]
for stage_index, stage in enumerate(model_classes):
    messages[f"stage_{stage}_probability"] = probabilities[:, stage_index]

probability_columns = [column for column in messages.columns if column.startswith("stage_") and column.endswith("_probability")]
for probability_column in probability_columns:
    positive = messages.loc[messages["is_predatory_line"] == 1, probability_column]
    negative = messages.loc[messages["is_predatory_line"] == 0, probability_column]
    y_true = messages["is_predatory_line"]
    print(f"{probability_column}: predatory mean={positive.mean():.4f}, other mean={negative.mean():.4f}")
    print(f"  PR-AUC={average_precision_score(y_true, messages[probability_column]):.4f}")

stage_200_probability: predatory mean=0.2090, other mean=0.4404
  PR-AUC=0.0018
stage_600_probability: predatory mean=0.5398, other mean=0.3105
  PR-AUC=0.0176
stage_900_probability: predatory mean=0.2512, other mean=0.2491
  PR-AUC=0.0045


## Compare predicted stage evidence

ROC-AUC alone can be misleading under imbalance. PR-AUC and class means are reported for each stage probability; results do not imply that a stage code equals predator identity.

In [4]:
rows = []
for probability_column in probability_columns:
    scores = messages[probability_column].to_numpy()
    y_true = messages["is_predatory_line"].to_numpy()
    rows.append({
        "stage_probability": probability_column,
        "predatory_mean": messages.loc[y_true == 1, probability_column].mean(),
        "other_mean": messages.loc[y_true == 0, probability_column].mean(),
        "pr_auc": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
    })

stage_comparison = pd.DataFrame(rows)
stage_distribution = pd.crosstab(
    messages["predicted_stage"], messages["is_predatory_line"], normalize="columns"
).rename(columns={0: "other_share", 1: "predatory_share"})

display(stage_comparison)
display(stage_distribution)

,stage_probability,predatory_mean,other_mean,pr_auc,roc_auc
0,stage_200_probability,0.208965,0.440398,0.001810,0.220072
1,stage_600_probability,0.539824,0.310470,0.017554,0.752373
2,stage_900_probability,0.251212,0.249132,0.004513,0.467139


is_predatory_line,other_share,predatory_share
predicted_stage,,
200,0.515448,0.143408
600,0.289284,0.659617
900,0.195268,0.196974


## Save validation artifacts

In [5]:
messages.to_csv(PROCESSED_DIR / "pan12_test_stage1_predictions.csv", index=False)
stage_comparison.to_csv(REPORT_DIR / "stage1_pan12_external_comparison.csv", index=False)
stage_distribution.to_csv(REPORT_DIR / "stage1_pan12_predicted_distribution.csv")

print("Saved PAN-12 Stage 1 external validation artifacts.")

Saved PAN-12 Stage 1 external validation artifacts.
